### 🛠️ Step 2: Constructing the Final Analytics Layer (One Big Table Architecture)
**Objective:** Join the central `master_sales` fact table with all cleansed dimension tables to build a single, comprehensive reporting view (`pnc_report`). Calculate row-level financials (Net Cost, Net Sale, Profit) directly in the data layer.

**Architectural Benefit:** * **Engine Optimization:** Pre-calculating joins in Spark (using an OBT approach) removes the relational computational burden from Power BI's VertiPaq engine, ensuring zero-latency dashboard rendering.
* **Data Quality Failsafe:** Implements explicit deduplication (`dropDuplicates`) and a strict Delta table overwrite protocol ("Nuke and Pave") to prevent transaction log bloat and guarantee absolute accuracy in the final reporting numbers.
m

#### Lets Load all  the file which will be connected to main master file

### ERM Diagram for above uploaded file

## The required columns in final table from all file is:
- OrderDate, Customer_Name, Gender, 
- Region, Country, Product_Name, product_categories, 
- Cost_Price, Sale_Price, OrderQuantity, Net_Cost, Net_Sale, Profit

In [1]:
QUERY = ("""
SELECT
    ms.OrderNumber,
    to_date(ms.OrderDate, 'yyyy-mm-dd') as OrderDate,
    YEAR(ms.OrderDate) as Order_Year,
    date_format(ms.OrderDate, 'yyyy-MM') as Order_Month,
    coalesce(concat(cl.FirstName,' ', cl.LastName), 'Walk-in-Customer') as Customer_Name,
    coalesce(cl.Gender, 'N/A') as Gender,
    tl.Region,
    tl.Country,
    pm.ProductName,
    pm.CategoryName,
    pm.SubcategoryName,
    pm.ModelName,
    pm.ProductCost as Cost_Price,
    pm.ProductPrice as Sale_Price,
    ms.OrderQuantity,
    round(pm.ProductCost * ms.orderquantity, 2) as Net_Cost,
    round(pm.productprice * ms.orderquantity, 2)as Net_Sale,
    round(((pm.productprice - pm.ProductCost) * ms.orderquantity),2) as Profit,
    round((pm.productprice - pm.ProductCost)*100/(pm.productprice), 2) as Profit_Margin
from ankush_schema.master_sales ms

left join ankush_schema.`territory lookup` tl
on ms.TerritoryKey = tl.SalesTerritoryKey

left join ankush_schema.`customer_lookup` cl
on ms.CustomerKey = cl.CustomerKey

left join ankush_schema.product_master pm
on pm.ProductKey = ms.ProductKey
""")

StatementMeta(, 75df8ee0-1235-4e22-9fbb-5b1e3b53e560, 3, Finished, Available, Finished, False)

In [2]:
df = spark.sql(QUERY)

StatementMeta(, 75df8ee0-1235-4e22-9fbb-5b1e3b53e560, 4, Finished, Available, Finished, False)

In [9]:
# 1. Nuke the old table completely out of existence
spark.sql("DROP TABLE IF EXISTS ankush_schema.pnc_report")



StatementMeta(, 75df8ee0-1235-4e22-9fbb-5b1e3b53e560, 11, Finished, Available, Finished, False)

DataFrame[]

In [10]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("ankush_schema.pnc_report")


StatementMeta(, 75df8ee0-1235-4e22-9fbb-5b1e3b53e560, 12, Finished, Available, Finished, False)

In [11]:
df.count()

StatementMeta(, 75df8ee0-1235-4e22-9fbb-5b1e3b53e560, 13, Finished, Available, Finished, False)

1169

In [12]:
df.distinct().count()

StatementMeta(, 75df8ee0-1235-4e22-9fbb-5b1e3b53e560, 14, Finished, Available, Finished, False)

1169